# linkedin-content-agent + character-forge-v2: full pipeline on Colab

Clones both repos as true siblings (matching the local `subprocess.Popen`-based architecture), brings up Ollama + ComfyUI + the FastAPI backend, then drives the real brief -> clarify -> draft/score -> lock -> image pipeline through the JSON API directly (no need to tunnel the React frontend for this test).

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine).

Repos:
- https://github.com/digishgabhawala/character-forge-v2
- https://github.com/digishgabhawala/linkedin-content-agent

## 1. Clone both repos as siblings

linkedin-content-agent's `config.py` auto-resolves `character_forge_v2_path` as `../character-forge-v2` relative to itself -- both repos need to sit side by side under the same parent directory for that to resolve correctly, exactly like on a local machine.

In [ ]:
%cd /content
!git clone https://github.com/digishgabhawala/character-forge-v2.git
!git clone https://github.com/digishgabhawala/linkedin-content-agent.git

## 2. character-forge-v2 side: ComfyUI + forge2 + models

Reuses `character-forge-v2/scripts/colab_bootstrap.sh` -- the same script that repo's own demo notebook uses -- rather than duplicating these steps here. Takes 15-30+ min the first time (models are ~34GB); set `USE_DRIVE = True` below to persist them across sessions.

In [ ]:
USE_DRIVE = False  # set True to persist model downloads across sessions --
                    # also keeps the ~34GB model set (and, if set, Ollama's
                    # own ~9GB qwen3:14b weights) OFF the local VM disk
                    # entirely, since Drive is FUSE-mounted cloud storage,
                    # not local disk quota. Recommended if you hit "disk
                    # almost full" warnings on Colab's ~107GB standard disk
                    # (this happened during live testing).

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODELS_DIR = "/content/drive/MyDrive/character-forge-v2-models"
    OLLAMA_MODELS_DIR = "/content/drive/MyDrive/ollama-models"
else:
    MODELS_DIR = "/content/ComfyUI/models"
    OLLAMA_MODELS_DIR = None  # Ollama's own default (local disk)

print("Models will be stored at:", MODELS_DIR)

In [ ]:
%cd /content/character-forge-v2
!bash scripts/colab_bootstrap.sh /content/ComfyUI "{MODELS_DIR}"

In [ ]:
import subprocess
import sys

with open("/content/comfyui.log", "w") as comfy_log:
    comfy_proc = subprocess.Popen(
        [sys.executable, "/content/ComfyUI/main.py"],
        stdout=comfy_log, stderr=subprocess.STDOUT,
    )
print("ComfyUI starting (pid", comfy_proc.pid, ") -- log at /content/comfyui.log")

In [ ]:
!python scripts/wait_for_comfyui.py http://127.0.0.1:8188 240

## 3. Ollama + qwen3:14b

linkedin-content-agent's clarify/draft/judge/scene calls all go through Ollama, not ComfyUI -- this is a second, separate service in the same VM. The `qwen3:14b` pull is another substantial download (~9GB).

### Before pulling qwen3:14b: free up disk space

Found live during testing: Colab's ~107GB standard disk was at 90.69GB/112.64GB used at this point in the notebook -- pip/apt build caches from the ComfyUI install add up on top of the ~34GB model set. Safe cleanup first (reclaims caches only, nothing here touches anything still needed):

In [ ]:
!apt-get clean
!pip cache purge
!rm -rf /content/sample_data
!df -h /content

Also found live: Ollama's install script needs `zstd` to decompress itself, which Colab's base image doesn't ship:

In [ ]:
!apt-get -qq install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess

env = os.environ.copy()
if OLLAMA_MODELS_DIR:
    os.makedirs(OLLAMA_MODELS_DIR, exist_ok=True)
    env["OLLAMA_MODELS"] = OLLAMA_MODELS_DIR

with open("/content/ollama.log", "w") as ollama_log:
    ollama_proc = subprocess.Popen(["ollama", "serve"], stdout=ollama_log, stderr=subprocess.STDOUT, env=env)
print("Ollama starting (pid", ollama_proc.pid, ") -- log at /content/ollama.log")
if OLLAMA_MODELS_DIR:
    print("Model storage redirected to:", OLLAMA_MODELS_DIR)

In [ ]:
import time

import requests

for _ in range(60):
    try:
        if requests.get("http://127.0.0.1:11434", timeout=3).status_code == 200:
            print("Ollama is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Ollama did not come up -- check /content/ollama.log")

In [ ]:
!ollama pull qwen3:14b

**Known unknown, worth watching:** ComfyUI and Ollama both compete for the same T4 VRAM/system RAM when both are loaded -- on the original Mac dev machine this caused silently-corrupted image renders (see character-forge-v2's README Known Issues). This notebook never calls both at the same time within a single pipeline run (Ollama drives clarify/draft/scene *before* the image render starts), but if you re-run cells out of order, keep that contention in mind.

## 4. linkedin-content-agent backend

Installed into its own venv, kept separate from the system Python that ComfyUI/forge2 live in -- the same convention local dev uses (`backend/.venv` alongside the sibling `comfyui-env`). Found live: installing this backend's pinned dependencies (fastapi/uvicorn/starlette/httpx) directly into Colab's system Python produced real pip dependency-conflict warnings against unrelated preinstalled packages (google-genai, gradio, google-adk, firebase-admin, python-fasthtml) that this project never imports -- a venv avoids that entirely.

In [ ]:
%cd /content/linkedin-content-agent/backend
!python3 -m venv .venv
!.venv/bin/pip install -q -r requirements.txt

In [ ]:
import sys

# sys.executable here is Colab's system Python (where forge2/ComfyUI
# were installed in section 2) -- NOT the backend/.venv just created
# above, which only holds this backend's own dependencies.
with open(".env", "w") as f:
    f.write(f"COMFYUI_ENV_PYTHON={sys.executable}\n")

with open(".env") as f:
    print(f.read())

In [ ]:
import subprocess

with open("/content/backend.log", "w") as backend_log:
    backend_proc = subprocess.Popen(
        [".venv/bin/python", "-m", "uvicorn", "app.main:app", "--port", "11000"],
        stdout=backend_log, stderr=subprocess.STDOUT,
    )
print("Backend starting (pid", backend_proc.pid, ") -- log at /content/backend.log")

In [ ]:
import time

import requests

for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:11000/api/health", timeout=3).status_code == 200:
            print("Backend is up.")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("Backend did not come up -- check /content/backend.log")

## 5. Drive the real pipeline through the JSON API

No frontend/tunnel needed to validate the actual brief -> post -> image loop -- everything the React UI does is just calls to this same API. (The frontend itself was already validated locally; this notebook is about proving *deployability*, not re-testing the UI.)

Edit `BRIEF` below to whatever you want to test with -- a richer brief (specific numbers, what was tried, why it mattered) is more likely to sail through scoring without a clarify/escalation round-trip; a thin one is a good test of the clarify + thin-material-gate behavior described in the main README.

In [ ]:
import requests

API = "http://127.0.0.1:11000/api"

BRIEF = (
    "Just open-sourced two projects: a local character-illustration generator "
    "(character-forge-v2, ComfyUI + Qwen-Image-Edit, no cloud, no LoRA training) "
    "and this LinkedIn content agent itself, which grades its own drafts against "
    "explicit quality gates before a human ever sees them -- factual integrity and "
    "voice authenticity are hard gates, plus hook/structure/length/etc as "
    "recalibrated optimization pillars. Both repos are public on GitHub now."
)

post = requests.post(f"{API}/posts", json={"brief": BRIEF}).json()
print("post id:", post["id"], "| status:", post["status"])
post_id = post["id"]

In [ ]:
# Clarify loop: answer questions until it leaves "clarifying". Run this cell
# repeatedly (or wrap it in a while loop) if it asks more than one question.
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])

if post["status"] == "clarifying" and post["pending_question"]:
    print("\nQuestion:", post["pending_question"])
    answer = input("Your answer: ")
    post = requests.post(f"{API}/posts/{post_id}/clarify", json={"answer": answer}).json()
    print("\nnew status:", post["status"])
else:
    print("No pending question -- re-run this cell later if status is still 'clarifying',\n"
          "otherwise move on to the next cell.")

**If status is `needs_input`**: a scoring gate that a rewrite can't fix (not enough material, or the wrong angle) escalated back to you -- see `post["escalation_reason"]`. Either provide more detail (`POST /posts/{id}/additional-info {"info": "..."}`) or accept the current best draft as-is (`POST /posts/{id}/accept-draft`, no body).

In [ ]:
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])

if post["status"] == "needs_input":
    print("\nEscalation reason:", post["escalation_reason"])
    choice = input("Type more detail to add, or leave blank to accept the current draft as-is: ")
    if choice.strip():
        post = requests.post(f"{API}/posts/{post_id}/additional-info", json={"info": choice}).json()
    else:
        post = requests.post(f"{API}/posts/{post_id}/accept-draft", json={}).json()
    print("\nnew status:", post["status"])
else:
    print("Not in needs_input -- nothing to do here.")

In [ ]:
post = requests.get(f"{API}/posts/{post_id}").json()
print("status:", post["status"])
print("category:", post["category"], "| weighted_score:", post["weighted_score"])
print("\n--- draft ---\n")
print(post["post_text"])
print("\n--- pillar scores ---")
for pillar, s in {**post["gate_scores"], **post["pillar_scores"]}.items():
    print(f"  {pillar}: {s['score']}/10 -- {s['reason']}")

## 6. Lock and generate the image

Locking derives a scene description (and reuses/creates an `@name` recurring backdrop asset if the scene proposes one). The render itself is a real 15-40 min wait on a T4 -- this cell polls until it's done.

In [ ]:
post = requests.post(f"{API}/posts/{post_id}/lock", json={}).json()
print("status:", post["status"])
print("scene_instruction:", post["scene_instruction"])
print("scene_asset_name:", post["scene_asset_name"])

In [ ]:
post = requests.post(f"{API}/posts/{post_id}/generate-image", json={}).json()
print("status:", post["status"])

In [ ]:
import time

start = time.time()
while True:
    post = requests.get(f"{API}/posts/{post_id}").json()
    elapsed = int(time.time() - start)
    print(f"[{elapsed}s] status: {post['status']}")
    if post["status"] in ("image_ready", "image_failed"):
        break
    time.sleep(30)

if post["status"] == "image_failed":
    print("\nFAILED:", post["image_job_error"])
else:
    print("\nDone.")

In [ ]:
from IPython.display import Image, display

if post["status"] == "image_ready":
    print(post["post_text"])
    display(Image(url=f"http://127.0.0.1:11000{post['image_url']}"))

## What this proves (and doesn't)

**Proves:** both repos, cloned fresh as siblings on a machine that isn't the original dev Mac, can actually run the full brief -> clarify -> scored draft -> locked scene -> rendered image loop end to end -- not just "each repo boots on its own," the real `subprocess.Popen` handoff between them.

**Doesn't cover:** the React frontend itself (this notebook drives the API directly -- the UI was already validated locally against this same API contract), the `lightning` speed profile, or feedback consumption (not built yet, see the README's Upcoming section).